# Task B — v0 abundance-only ranker

Consumes the processed artefacts from `01_data.ipynb` and trains a per-family classifier that maps a family's abundance profile (1638-dim vector across HMP2 metagenomic samples) to a bioactivity score. The trained model produces a ranking that we evaluate with AUPRC / AUROC / Precision@K.

**Hard target to beat:** MetaWIBELE unsupervised priority AUPRC = 0.0779 (from `processed/summary.txt`). The supervised version sits at 0.0675. Random baseline = 0.0187 (base rate).

**Why this is interesting:** MetaWIBELE's unsupervised score is essentially `harmonic_mean(prevalence, mean_abundance)`. We're asking whether a learned non-linear function of the *full* abundance profile (which preserves sample-level structure: who has it, in what amounts, in which disease state) extracts more signal than the two aggregate statistics.

**Comparison stack on the same test split:**
1. Random baseline
2. Ecology score (mean abundance + prevalence on train samples) — reproduces MetaWIBELE-unsup logic on our split
3. MetaWIBELE unsupervised (their published rank, evaluated on our test families only)
4. MetaWIBELE supervised   (their published rank, evaluated on our test families only)
5. Linear classifier on abundance (LogReg with L2)
6. MLP on abundance (this notebook's main model)

**Compute budget:** ~5-15 min on MPS. The abundance matrix is mmap'd so we don't blow RAM.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

torch.manual_seed(0); np.random.seed(0)

PROC = Path('processed')
PRED = Path('predictions'); PRED.mkdir(exist_ok=True)

device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

device: mps


In [2]:
X = np.load(PROC / 'abundance.npy', mmap_mode='r')  # mmap — no RAM cost
labels = pd.read_parquet(PROC / 'labels.parquet')
fids = pd.read_parquet(PROC / 'family_ids.parquet')['family_id'].values
sample_ids = json.load(open(PROC / 'sample_ids.json'))

assert len(fids) == X.shape[0], (len(fids), X.shape)
assert (labels['family_id'].values == fids).all(), 'family order mismatch between labels and abundance'

y = labels['is_bioactive'].astype(np.int8).values
base_rate = y.mean()
print(f'X (mmap): {X.shape}  dtype={X.dtype}')
print(f'y:        {y.shape}  positives={int(y.sum()):,}  base rate={base_rate:.4f}')
print(f'samples:  {len(sample_ids)}  (example: {sample_ids[:3]} ...)')

X (mmap): (1447952, 1595)  dtype=float32
y:        (1447952,)  positives=27,034  base rate=0.0187
samples:  1595  (example: ['CSM5FZ3N_P', 'CSM5FZ3R_P', 'CSM5FZ3T_P'] ...)


## Train / val / test split

We split on **families** (rows), not samples. Each family is one prediction target. Stratified by `is_bioactive` so all three splits keep the 1.87% base rate. 70 / 15 / 15.

In [3]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.15, stratify=y, random_state=0)
tr, va = train_test_split(tr,  test_size=0.15 / 0.85, stratify=y[tr], random_state=0)
for name, ix in [('train', tr), ('val', va), ('test', te)]:
    print(f'  {name:5s}: {len(ix):>8,}   pos={int(y[ix].sum()):>6,}   rate={y[ix].mean():.4f}')

  train: 1,013,566   pos=18,924   rate=0.0187
  val  :  217,193   pos= 4,055   rate=0.0187
  test :  217,193   pos= 4,055   rate=0.0187


## Baselines 1-4: random, ecology score, MetaWIBELE unsup, MetaWIBELE sup

All evaluated on the **same test families** so the comparison is apples-to-apples.

In [4]:
def report(name: str, scores: np.ndarray, y_true: np.ndarray, K=(100, 1000)) -> dict:
    auprc = average_precision_score(y_true, scores)
    auroc = roc_auc_score(y_true, scores)
    out = {'method': name, 'auprc': auprc, 'auroc': auroc}
    for k in K:
        top = np.argpartition(-scores, k)[:k]
        p_at_k = y_true[top].mean()
        out[f'P@{k}'] = p_at_k
        out[f'enr@{k}'] = p_at_k / max(y_true.mean(), 1e-9)
    return out

results = []
y_te = y[te]

# 1. Random.
rng = np.random.default_rng(0)
results.append(report('Random', rng.random(len(te)), y_te))

# 2. Ecology score on our split: mean(abundance) + prevalence, fitted on TRAIN samples only.
#    The MetaWIBELE unsupervised priority is essentially harmonic_mean(prevalence, mean_abundance);
#    reproducing it on our split gives a fair sanity check + isolates the value of going beyond it.
#    Compute per-family stats once via mmap row reads.
def per_family_stats(idx: np.ndarray, batch=8192) -> tuple[np.ndarray, np.ndarray]:
    means = np.empty(len(idx), dtype=np.float32)
    prevs = np.empty(len(idx), dtype=np.float32)
    for s in range(0, len(idx), batch):
        rows = X[idx[s:s+batch]]
        means[s:s+batch] = rows.mean(axis=1)
        prevs[s:s+batch] = (rows > 0).mean(axis=1)
    return means, prevs

print('computing per-family abundance stats on test set ...')
te_means, te_prevs = per_family_stats(te)
eco_score = 2 / (1/(te_means + 1e-9) + 1/(te_prevs + 1e-9))  # harmonic mean
results.append(report('Ecology score (harm.mean abu,prev)', eco_score, y_te))

# 3-4. MetaWIBELE published priorities, evaluated on the same test families.
mw_unsup = labels['metawibele_unsup_rank'].fillna(0).values[te]
mw_sup = labels['metawibele_sup_rank'].fillna(0).values[te]
results.append(report('MetaWIBELE unsup (published)', mw_unsup, y_te))
results.append(report('MetaWIBELE sup   (published)', mw_sup, y_te))

for r in results: print(f'  {r["method"]:40s} AUPRC={r["auprc"]:.4f}  AUROC={r["auroc"]:.4f}  P@100={r["P@100"]:.3f} (enr={r["enr@100"]:.1f}x)  P@1000={r["P@1000"]:.3f} (enr={r["enr@1000"]:.1f}x)')

computing per-family abundance stats on test set ...


  Random                                   AUPRC=0.0184  AUROC=0.4973  P@100=0.000 (enr=0.0x)  P@1000=0.012 (enr=0.6x)
  Ecology score (harm.mean abu,prev)       AUPRC=0.0751  AUROC=0.7462  P@100=0.350 (enr=18.7x)  P@1000=0.210 (enr=11.2x)
  MetaWIBELE unsup (published)             AUPRC=0.0755  AUROC=0.7296  P@100=0.350 (enr=18.7x)  P@1000=0.212 (enr=11.4x)
  MetaWIBELE sup   (published)             AUPRC=0.0640  AUROC=0.7029  P@100=0.320 (enr=17.1x)  P@1000=0.198 (enr=10.6x)


## Dataset / DataLoader

Reads rows directly from the mmap'd abundance matrix, applies `log1p` and per-feature standardization (fit on train means/stds), and yields `(X_batch, y_batch)`. Workers stay at 0 because the OS already pages the mmap efficiently and torch DataLoader workers + mmap don't play well together on macOS.

In [5]:
print('computing per-feature mean/std on TRAIN rows (for standardization) ...')
n_feat = X.shape[1]
feat_sum = np.zeros(n_feat, dtype=np.float64)
feat_sq = np.zeros(n_feat, dtype=np.float64)
B = 16_384
tr_sorted = np.sort(tr)  # sequential mmap access is much faster than scattered
for s in range(0, len(tr_sorted), B):
    rows = np.log1p(X[tr_sorted[s:s+B]].astype(np.float32))
    feat_sum += rows.sum(axis=0)
    feat_sq += (rows ** 2).sum(axis=0)
feat_mean = (feat_sum / len(tr)).astype(np.float32)
feat_std = (np.sqrt(np.maximum(feat_sq / len(tr) - feat_mean ** 2, 1e-12))).astype(np.float32)
print(f'  feat_mean range: [{feat_mean.min():.3f}, {feat_mean.max():.3f}]')
print(f'  feat_std  range: [{feat_std.min():.3f}, {feat_std.max():.3f}]')

class FamilyDataset(Dataset):
    def __init__(self, idx: np.ndarray):
        self.idx = idx
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        row_i = self.idx[i]
        x = np.log1p(X[row_i].astype(np.float32))
        x = (x - feat_mean) / feat_std
        return torch.from_numpy(x), torch.tensor(y[row_i], dtype=torch.float32)

BATCH = 4096
tr_dl = DataLoader(FamilyDataset(tr), batch_size=BATCH, shuffle=True,  num_workers=0)
va_dl = DataLoader(FamilyDataset(va), batch_size=BATCH, shuffle=False, num_workers=0)
te_dl = DataLoader(FamilyDataset(te), batch_size=BATCH, shuffle=False, num_workers=0)
print(f'batches per epoch: train={len(tr_dl)}  val={len(va_dl)}  test={len(te_dl)}')

computing per-feature mean/std on TRAIN rows (for standardization) ...


  feat_mean range: [0.012, 0.223]
  feat_std  range: [0.206, 0.569]
batches per epoch: train=248  val=54  test=54


## Models 5-6: linear classifier + MLP

Same training loop (BCEWithLogits + pos-weight for the 1.87% class imbalance, Adam, early stop on val AUPRC). The linear model is `Linear(1638, 1)`; the MLP is `Linear(1638, 256) -> ReLU -> Dropout(0.3) -> Linear(256, 64) -> ReLU -> Linear(64, 1)`.

In [6]:
pos_weight = torch.tensor([(len(tr) - y[tr].sum()) / max(y[tr].sum(), 1)], dtype=torch.float32, device=device)
print(f'pos_weight (negatives/positives in train) = {pos_weight.item():.2f}')

def predict(model, dl) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for xb, _ in dl:
            out.append(torch.sigmoid(model(xb.to(device))).squeeze(-1).cpu().numpy())
    return np.concatenate(out)

def train(model, name: str, epochs=8, lr=1e-3, weight_decay=1e-5):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_va = -1.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        t0 = time.time(); seen = 0; loss_sum = 0.0
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb).squeeze(-1)
            loss = loss_fn(logits, yb)
            loss.backward(); opt.step()
            loss_sum += loss.item() * len(xb); seen += len(xb)
        va_scores = predict(model, va_dl)
        va_auprc = average_precision_score(y[va], va_scores)
        print(f'  ep {ep}/{epochs}  loss={loss_sum/seen:.4f}  val AUPRC={va_auprc:.4f}  ({time.time()-t0:.1f}s)')
        if va_auprc > best_va:
            best_va = va_auprc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_va

print('\n[5/6] Linear classifier on abundance ...')
linear = nn.Linear(n_feat, 1)
linear, best_va_lin = train(linear, 'linear', epochs=6, lr=3e-3, weight_decay=1e-4)
lin_scores = predict(linear, te_dl)
results.append(report('Linear (LogReg on abundance)', lin_scores, y_te))

print('\n[6/6] MLP on abundance ...')
class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x)
mlp = MLP(n_feat)
mlp, best_va_mlp = train(mlp, 'mlp', epochs=8, lr=1e-3, weight_decay=1e-5)
mlp_scores = predict(mlp, te_dl)
results.append(report('MLP on abundance', mlp_scores, y_te))

pos_weight (negatives/positives in train) = 52.56

[5/6] Linear classifier on abundance ...


  ep 1/6  loss=1.1778  val AUPRC=0.1077  (21.4s)


  ep 2/6  loss=1.1100  val AUPRC=0.1070  (20.2s)


  ep 3/6  loss=1.1248  val AUPRC=0.1044  (20.3s)


  ep 4/6  loss=1.0953  val AUPRC=0.0876  (20.1s)


...

  ep 4/8  loss=0.9902  val AUPRC=0.1399  (22.5s)


  ep 5/8  loss=0.9749  val AUPRC=0.1358  (22.6s)


  ep 6/8  loss=0.9643  val AUPRC=0.1402  (22.6s)


  ep 7/8  loss=0.9518  val AUPRC=0.1439  (22.9s)


  ep 8/8  loss=0.9444  val AUPRC=0.1388  (23.4s)


## Results table

In [7]:
res_df = pd.DataFrame(results).set_index('method').round(4)
print(res_df.to_string())
best = res_df['auprc'].idxmax()
print(f'\nbest AUPRC on test: {best} ({res_df.loc[best, "auprc"]:.4f})')
mw_target = res_df.loc['MetaWIBELE unsup (published)', 'auprc']
print(f'MetaWIBELE-unsup published target on this split: {mw_target:.4f}')
for m in ['Linear (LogReg on abundance)', 'MLP on abundance']:
    if m in res_df.index:
        delta = res_df.loc[m, 'auprc'] - mw_target
        sign = '+' if delta >= 0 else ''
        print(f'  {m}: {sign}{delta:.4f} vs MetaWIBELE')

                                     auprc   auroc  P@100  enr@100  P@1000  enr@1000
method                                                                              
Random                              0.0184  0.4973   0.00   0.0000   0.012    0.6427
Ecology score (harm.mean abu,prev)  0.0751  0.7462   0.35  18.7466   0.210   11.2480
MetaWIBELE unsup (published)        0.0755  0.7296   0.35  18.7466   0.212   11.3551
MetaWIBELE sup   (published)        0.0640  0.7029   0.32  17.1398   0.198   10.6052
Linear (LogReg on abundance)        0.1040  0.7970   0.34  18.2110   0.264   14.1403
MLP on abundance                    0.1424  0.8315   0.41  21.9603   0.339   18.1574

best AUPRC on test: MLP on abundance (0.1424)
MetaWIBELE-unsup published target on this split: 0.0755
  Linear (LogReg on abundance): +0.0285 vs MetaWIBELE
  MLP on abundance: +0.0669 vs MetaWIBELE


## Stratified eval by characterization category

The headline thesis question is whether DL helps more for *uncharacterized* (novel) protein families. Using `annotations.parquet`, we bucket each family into:
- **characterized**: has a strong UniRef90 homology hit
- **weakly characterized**: only a weak homology hit
- **novel (NH)**: no homology hit — the families MetaWIBELE was specifically designed to surface

If our MLP beats MetaWIBELE more on the *novel* bucket than on the *characterized* one, that's a real research result.

In [8]:
anno = pd.read_parquet(PROC / 'annotations.parquet')
anno['bucket'] = np.where(anno['strong_homology'].notna(), 'characterized',
                  np.where(anno['weak_homology'].notna(), 'weak', 'novel'))
fam_to_bucket = dict(zip(anno['family_id'], anno['bucket']))
te_buckets = np.array([fam_to_bucket.get(fid, 'novel') for fid in fids[te]])
print('test set composition by bucket:')
for b, n in pd.Series(te_buckets).value_counts().items():
    pos = int(y_te[te_buckets == b].sum())
    print(f'  {b:15s}  n={n:>6,}  pos={pos:>5,}  rate={pos/n:.4f}')

rows = []
for b in ['characterized', 'weak', 'novel']:
    mask = te_buckets == b
    if mask.sum() < 50 or y_te[mask].sum() < 5:
        continue
    row = {'bucket': b, 'n': int(mask.sum()), 'pos': int(y_te[mask].sum()),
           'base_rate': y_te[mask].mean()}
    for nm, sc in [('MW_unsup', mw_unsup), ('MW_sup', mw_sup), ('MLP', mlp_scores)]:
        try: row[f'{nm}_AUPRC'] = average_precision_score(y_te[mask], sc[mask])
        except ValueError: row[f'{nm}_AUPRC'] = np.nan
    rows.append(row)
strat = pd.DataFrame(rows).round(4)
print()
print(strat.to_string(index=False))

test set composition by bucket:
  characterized    n=125,750  pos=3,302  rate=0.0263
  weak             n=67,135  pos=  703  rate=0.0105
  novel            n=24,308  pos=   50  rate=0.0021

       bucket      n  pos  base_rate  MW_unsup_AUPRC  MW_sup_AUPRC  MLP_AUPRC
characterized 125750 3302     0.0263          0.0963        0.0791     0.1672
         weak  67135  703     0.0105          0.0239        0.0229     0.0590
        novel  24308   50     0.0021          0.0034        0.0028     0.0052


## Save predictions and result tables

Test-set predictions go to `predictions/02_abundance_only.parquet` so `03_sequence.ipynb` can fuse them with sequence-derived scores later (simple ensemble: rank-average or learned).

In [9]:
pred_df = pd.DataFrame({
    'family_id': fids[te],
    'y_true':    y_te,
    'mlp_score': mlp_scores,
    'lin_score': lin_scores,
    'eco_score': eco_score,
    'mw_unsup':  mw_unsup,
    'mw_sup':    mw_sup,
})
pred_df.to_parquet(PRED / '02_abundance_only.parquet', index=False)
res_df.to_csv(PRED / '02_results_table.csv')
strat.to_csv(PRED / '02_results_stratified.csv', index=False)
torch.save(mlp.state_dict(), PRED / '02_mlp.pt')
print(f'wrote {PRED}/02_abundance_only.parquet  ({(PRED/"02_abundance_only.parquet").stat().st_size/1e6:.2f} MB)')
print(f'wrote {PRED}/02_results_table.csv')
print(f'wrote {PRED}/02_results_stratified.csv')
print(f'wrote {PRED}/02_mlp.pt  ({(PRED/"02_mlp.pt").stat().st_size/1e6:.2f} MB)')

wrote predictions/02_abundance_only.parquet  (7.65 MB)
wrote predictions/02_results_table.csv
wrote predictions/02_results_stratified.csv
wrote predictions/02_mlp.pt  (1.70 MB)


## What's next

- If MLP beats MetaWIBELE on this split: that's the headline result for the abundance-only experiment. The stratified table tells you *where* the lift comes from.
- If MLP barely matches or underperforms: that's also a legit thesis finding — it means MetaWIBELE's two aggregate statistics already capture most of the predictive signal in the abundance matrix, and the value of DL must come from sequence (`03_sequence.ipynb`) or fusion.
- Either way, the predictions saved in `predictions/02_abundance_only.parquet` become the input for the ensembling step in `03_sequence.ipynb`.